In [ ]:
# Importing req libraries
import os
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import time

# Importing required modules from sklearn library
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importing harmless warninga
import warnings
warnings.filterwarnings('ignore')

# Notebook formatting
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [206]:
# Importing and viewing sample data

df = pd.read_csv('data.csv')
df.head()

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost
0,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008,no,delivered,3,1632.7206
1,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,5,640.1700
2,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1448.1700
3,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008,no,delivered,3,1486.5700
4,250.99,dhl,clothing,van,two day,north,foggy,256.7,37.02,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016,no,delivered,4,1394.5600


In [207]:
# Data volume

df.shape

(25000, 15)

In [208]:
# Data validation and data cleaning

# Dataframe for missing value & datatype
msg_data = pd.DataFrame({'Missing Values Percentage' : round(df.isnull().mean()*100,2), 'Data Type' : df.dtypes})

# Dataframe for 2 samples
sample = df.head(2).T

# Concating msg_data and sample
msg_data_sample = pd.concat([msg_data, sample], axis = 1)

# Renaming columns
msg_data_sample.rename(columns = {0: 'Sample_1', 1: 'Sample_2'}, inplace = True)

# Missing data sample
msg_data_sample

,Missing Values Percentage,Data Type,Sample_1,Sample_2
delivery_id,0.0,float64,250.99,250.99
delivery_partner,0.0,object,delhivery,xpressbees
package_type,0.0,object,automobile parts,cosmetics
vehicle_type,0.0,object,bike,ev van
delivery_mode,0.0,object,same day,express
region,0.0,object,west,central
weather_condition,0.0,object,clear,cold
distance_km,0.0,float64,297.0,89.6
package_weight_kg,0.0,float64,46.96,47.39
delivery_time_hours,0.0,object,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000002


In [209]:
# Checking Row Duplicates

if len(df[df.duplicated])>0:
    print(f'Row has duplicates')
else:
    print(f'Row has no dupliactes')

Row has no dupliactes


In [210]:
# Checking Column Duplicates

has_duplictate = df.shape[1] != df.T.drop_duplicates().T.shape[1]

if has_duplictate:
    print(f'Column has dupliactes')
else:
    print(f'Column has no duplicates')

Column has no duplicates


In [211]:
# Diff cols in the data

df.columns

Index(['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type',
       'delivery_mode', 'region', 'weather_condition', 'distance_km',
       'package_weight_kg', 'delivery_time_hours', 'expected_time_hours',
       'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost'],
      dtype='object')

In [212]:
# Checking Cat and Num Columns

cat_cols = [col for col in df.columns if df[col].dtype == 'O']
num_cols = [col for col in df.columns if df[col].dtype != 'O']

print(f'Categorical Columns: {cat_cols}')
print(f'\nNumerical Columns: {num_cols}')

Categorical Columns: ['delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition', 'delivery_time_hours', 'expected_time_hours', 'delayed', 'delivery_status']

Numerical Columns: ['delivery_id', 'distance_km', 'package_weight_kg', 'delivery_rating', 'delivery_cost']


In [213]:
#Checking cols with single unique value
single_unique_cols = [col for col in df.columns if df[col].nunique() == 1]

if len(single_unique_cols)>0:
    print(f'Single Unique Columns:{single_unique_cols}')
else:
    print(f'The df has no single unique cols')


The df has no single unique cols


In [214]:
# Checking cols with more than 30% data missing

mv_grt_30 = [col for col in df.columns if round(df[col].isnull().mean()*100, 2)>30]

if len(mv_grt_30)>0:
    print(f'Columns with missing values greater than 30:{mv_grt_30}')
else:
    print(f'No column has greater than 30% data missing')

No column has greater than 30% data missing


In [215]:
# Extract encoded hours

def extract_encoded_hours(time_str):
    """
    Dataset encodes HOURS in the fractional part of a fake timestamp.
    Example: 1970-01-01 00:00:00.000000008 -> 8 hours
    """
    return int(time_str.split('.')[-1])


# Create ETA target in minutes

df['actual_delivery_hours'] = df['delivery_time_hours'].apply(extract_encoded_hours)
df['expected_delivery_hours'] = df['expected_time_hours'].apply(extract_encoded_hours)

# Convert hours → minutes

df['actual_delivery_minutes'] = df['actual_delivery_hours'] * 60
df['expected_delivery_minutes'] = df['expected_delivery_hours'] * 60

# Final regression target

df['eta_error_minutes'] = (df['actual_delivery_minutes'] - df['expected_delivery_minutes'])

# Optional derived label (NOT for modeling)

df['is_delayed_calc'] = np.where(df['eta_error_minutes'] > 0, 'yes', 'no')

# Drop raw columns

df.drop(
    columns=['delivery_time_hours', 'expected_time_hours'],
    inplace=True
)


In [216]:
df.head().T

,0,1,2,3,4
delivery_id,250.99,250.99,250.99,250.99,250.99
delivery_partner,delhivery,xpressbees,shadowfax,dhl,dhl
package_type,automobile parts,cosmetics,groceries,electronics,clothing
vehicle_type,bike,ev van,truck,ev van,van
delivery_mode,same day,express,two day,same day,two day
region,west,central,east,east,north
weather_condition,clear,cold,rainy,cold,foggy
distance_km,297.0,89.6,273.5,269.7,256.7
package_weight_kg,46.96,47.39,26.89,12.69,37.02
delayed,no,no,no,no,no


The dataset stored delivery duration as diff timestamps with the actual hour value embedded in the fractional component. After validating the data behavior, I extracted the encoded hours, converted them to minutes, and used the difference between actual and expected delivery time as the ETA regression target.

In [217]:
# Validate dataset consistency

(df['delayed'] == df['is_delayed_calc']).value_counts()

True     23797
False     1203
Name: count, dtype: int64

“Approximately 5% of rows showed inconsistency between the provided delay flag and computed ETA deviation, indicating real-world labeling noise.”

In [218]:
# List of non-important features to remove

rmv_list = [
    'delivery_id',
    'delivery_status',
    'delayed',
    'is_delayed_calc',
    'actual_delivery_hours',
    'expected_delivery_hours',
    'delivery_rating'
]

# Consolidated list for removal

removal_list = list(set(rmv_list + mv_grt_30 + single_unique_cols))

# Count of the features to be removed

print('The no of features to remove:', len(removal_list))

# Count of columns before removing

print('The no of columns before removing:', df.shape[1])

#Removing the features from the dfframe

for feature in removal_list:
    del df[feature]

# Count of the features after removal

print('The no of columns after removal:', df.shape[1])

The no of features to remove: 7
The no of columns before removing: 19
The no of columns after removal: 12


“I removed identifiers and post-outcome variables to prevent data leakage, while retaining pre-delivery operational and environmental features.

In [219]:
# Checking data after cleaning

df.head().T

,0,1,2,3,4
delivery_partner,delhivery,xpressbees,shadowfax,dhl,dhl
package_type,automobile parts,cosmetics,groceries,electronics,clothing
vehicle_type,bike,ev van,truck,ev van,van
delivery_mode,same day,express,two day,same day,two day
region,west,central,east,east,north
weather_condition,clear,cold,rainy,cold,foggy
distance_km,297.0,89.6,273.5,269.7,256.7
package_weight_kg,46.96,47.39,26.89,12.69,37.02
delivery_cost,1632.7206,640.17,1448.17,1486.57,1394.56
actual_delivery_minutes,480,120,600,360,540


In [220]:
# Checking Cat and Num Columns

cat_cols = [col for col in df.columns if df[col].dtype == 'O']
num_cols = [col for col in df.columns if df[col].dtype != 'O']

print(f'Categorical Columns: {cat_cols}')
print(f'\nNumerical Columns: {num_cols}')

Categorical Columns: ['delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition']

Numerical Columns: ['distance_km', 'package_weight_kg', 'delivery_cost', 'actual_delivery_minutes', 'expected_delivery_minutes', 'eta_error_minutes']


In [221]:
# No of outliers and skewness

features = ['distance_km', 'package_weight_kg', 'delivery_cost', 'actual_delivery_minutes', 'expected_delivery_minutes', 'eta_error_minutes']

skew_groups = {
    "Left skewed": [],
    "Right skewed": [],
    "Both sides (heavy tails)": [],
    "Approximately symmetric": []
}

for feature in features:
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3-q1
    lower_fence = q1-1.5*iqr
    upper_fence = q3+1.5*iqr
    lf_outliers = len(df[df[feature]<lower_fence])
    uf_outliers = len(df[df[feature]>upper_fence])

    if lf_outliers > 0 and uf_outliers == 0:
        skew = "Left skewed"
    elif uf_outliers > 0 and lf_outliers == 0:
        skew = "Right skewed"
    elif uf_outliers > 0 and lf_outliers > 0:
        skew = "Both sides (heavy tails)"
    else:
        skew = "Approximately symmetric"

    print(
    f"Feature: {feature}\n"
    f"Outliers below lower fence: {lf_outliers}\n"
    f"Outliers above upper fence: {uf_outliers}\n"
    f"Skewness (IQR-based): {skew}\n\n"
    )

# Checking skewness
    skew_groups[skew].append(feature)


# Skewed columns

for k, v in skew_groups.items():
    print(f"{k}: {v}\n")

Feature: distance_km
Outliers below lower fence: 0
Outliers above upper fence: 0
Skewness (IQR-based): Approximately symmetric


Feature: package_weight_kg
Outliers below lower fence: 0
Outliers above upper fence: 0
Skewness (IQR-based): Approximately symmetric


Feature: delivery_cost
Outliers below lower fence: 0
Outliers above upper fence: 0
Skewness (IQR-based): Approximately symmetric


Feature: actual_delivery_minutes
Outliers below lower fence: 0
Outliers above upper fence: 203
Skewness (IQR-based): Right skewed


Feature: expected_delivery_minutes
Outliers below lower fence: 0
Outliers above upper fence: 0
Skewness (IQR-based): Approximately symmetric


Feature: eta_error_minutes
Outliers below lower fence: 0
Outliers above upper fence: 0
Skewness (IQR-based): Approximately symmetric


Left skewed: []

Right skewed: ['actual_delivery_minutes']

Both sides (heavy tails): []

Approximately symmetric: ['distance_km', 'package_weight_kg', 'delivery_cost', 'expected_delivery_minutes

In [222]:
# Correlation between diff numeric columns
corr = df.corr(numeric_only=True) # Not using churn col since it's a categorical column
corr

,distance_km,package_weight_kg,delivery_cost,actual_delivery_minutes,expected_delivery_minutes,eta_error_minutes
distance_km,1.000000,0.002311,0.990772,0.685883,0.045236,0.224596
package_weight_kg,0.002311,1.000000,0.099479,0.001242,0.001442,-0.000867
delivery_cost,0.990772,0.099479,1.000000,0.678911,-0.025480,0.288125
actual_delivery_minutes,0.685883,0.001242,0.678911,1.000000,0.039755,0.351996
expected_delivery_minutes,0.045236,0.001442,-0.025480,0.039755,1.000000,-0.921268
eta_error_minutes,0.224596,-0.000867,0.288125,0.351996,-0.921268,1.000000


In [223]:
# Final df for modelling

final_df = df.drop(['expected_delivery_minutes', 'actual_delivery_minutes', 'delivery_cost'], axis = 1)
final_df.head()

,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,eta_error_minutes
0,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,0
1,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,-60
2,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,-360
3,dhl,electronics,ev van,same day,east,cold,269.7,12.69,-120
4,dhl,clothing,van,two day,north,foggy,256.7,37.02,-420


Actual and expected delivery times were used only to derive the ETA deviation target. Since both are post-hoc or baseline quantities, they were removed from the feature set to prevent data leakage. Since delivery cost was almost perfectly correlated with distance, and distance is the underlying physical driver, I removed delivery cost to reduce redundancy and simplify the model.

## Preparing the Data for ML Model

In [224]:
# Dependecy split
x = final_df.drop(['eta_error_minutes'], axis = 1)
y = final_df[['eta_error_minutes']]

# Train Test Split
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state = 42, test_size = 0.2)

# Dimension of the further splits
print('DIMENSION OF THE FURTHER SPLIT:')
print('\nx_train data dimension:', x_train.shape)
print('y_train data dimension:', y_train.shape)
print('x_test data dimension:', x_test.shape)
print('y_test data dimension:', y_test.shape)

DIMENSION OF THE FURTHER SPLIT:

x_train data dimension: (20000, 8)
y_train data dimension: (20000, 1)
x_test data dimension: (5000, 8)
y_test data dimension: (5000, 1)


In [225]:
cat_cols = [col for col in df.columns if df[col].dtype == 'O']

In [226]:
# One Hot Encoding
encoder = OneHotEncoder(
    sparse_output=False,
    drop='first',
    handle_unknown='ignore'
)

# Training data
x_train_encoded = encoder.fit_transform(x_train[cat_cols])
x_train_encoded_df = pd.DataFrame(
    x_train_encoded,
    index=x_train.index,
    columns=encoder.get_feature_names_out()
)

x_train_encoded_df = pd.concat(
    [x_train.drop(cat_cols, axis=1), x_train_encoded_df],
    axis=1
)

# Test data (ONLY transform)
x_test_encoded = encoder.transform(x_test[cat_cols])
x_test_encoded_df = pd.DataFrame(
    x_test_encoded,
    index=x_test.index,
    columns=encoder.get_feature_names_out()
)

x_test_encoded_df = pd.concat(
    [x_test.drop(cat_cols, axis=1), x_test_encoded_df],
    axis=1
)

In [228]:
# Data scaling for independent columns (AFTER encoding)

scaler = MinMaxScaler()

# Train scaling (USE ENCODED DF)
x_train_scaled = scaler.fit_transform(x_train_encoded_df)
x_train_scaled_df = pd.DataFrame(
    x_train_scaled,
    index=x_train_encoded_df.index,
    columns=x_train_encoded_df.columns
)

# Test scaling (TRANSFORM ONLY)
x_test_scaled = scaler.transform(x_test_encoded_df)
x_test_scaled_df = pd.DataFrame(
    x_test_scaled,
    index=x_test_encoded_df.index,
    columns=x_test_encoded_df.columns
)

In [230]:
# Model
model = LinearRegression()

# Train on SCALED + ENCODED data
model.fit(x_train_scaled_df, y_train.values.ravel())

# Predict on SCALED + ENCODED test data
y_pred = model.predict(x_test_scaled_df)

# Metrics (in real units: minutes)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f} minutes")
print(f"RMSE: {rmse:.2f} minutes")
print(f"R²: {r2:.4f}")

MAE: 83.36 minutes
RMSE: 105.68 minutes
R²: 0.9521


In [231]:
# Compare MAE: Baseline vs ML

# Baseline MAE
baseline_mae = np.mean(np.abs(y_test))

# ML MAE
ml_mae = mean_absolute_error(y_test, y_pred)

# Improvement %
improvement = (baseline_mae - ml_mae) / baseline_mae * 100
print(f"ETA error reduced by {improvement:.2f}% compared to baseline")

ETA error reduced by 82.84% compared to baseline


In [232]:
baseline_mae = np.mean(np.abs(y_test))
ml_mae = mean_absolute_error(y_test, y_pred)

print("Baseline MAE (min):", baseline_mae)
print("ML MAE (min):", ml_mae)

Baseline MAE (min): 485.868
ML MAE (min): 83.36473244304445


Our baseline ETA system had an average error of over 8 hours due to fixed expected times. The ML model reduced this error to roughly 83 minutes by learning systematic deviations based on operational factors.

In [240]:
# Ensure same shape
y_test_arr = y_test.values.ravel()
y_pred_arr = np.array(y_pred)

sla_windows = [30, 60, 120]

print("Baseline SLA metrics:\n")
for window in sla_windows:
    baseline_sla = np.mean(np.abs(y_test_arr) <= window)
    print(f"SLA ±{window} min: {baseline_sla:.2%}")

print("\n\nML SLA metrics:\n")
for window in sla_windows:
    ml_sla = np.mean(np.abs(y_test_arr - y_pred_arr) <= window)
    print(f"SLA ±{window} min: {ml_sla:.2%}")

print("\n\nSLA Improvement:\n")
for window in sla_windows:
    baseline_sla = np.mean(np.abs(y_test_arr) <= window)
    ml_sla = np.mean(np.abs(y_test_arr - y_pred_arr) <= window)
    improvement = (ml_sla - baseline_sla) * 100
    print(f"SLA ±{window} min improved by {improvement:.2f} percentage")

Baseline SLA metrics:

SLA ±30 min: 6.80%
SLA ±60 min: 18.96%
SLA ±120 min: 28.04%


ML SLA metrics:

SLA ±30 min: 23.54%
SLA ±60 min: 44.72%
SLA ±120 min: 74.52%


SLA Improvement:

SLA ±30 min improved by 16.74 percentage
SLA ±60 min improved by 25.76 percentage
SLA ±120 min improved by 46.48 percentage


Compared to the existing ETA baseline, the ML model improved delivery time reliability across all SLA windows. The percentage of deliveries within ±60 minutes increased from 18.9% to 44.7%, and within ±120 minutes from 28.0% to 74.5%, demonstrating a substantial improvement in ETA accuracy.

Since the features showed low multicollinearity, regularization was unlikely to yield significant gains. Linear regression primarily underfit the non-linear structure of ETA deviations, so I moved to tree-based models instead.

## Using RF Regressor

In [241]:
# RF Model
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

# Train on SCALED + ENCODED data
rf_model.fit(x_train_scaled_df, y_train.values.ravel())

# Predict on SCALED + ENCODED test data
y_pred = rf_model.predict(x_test_scaled_df)

# Metrics (in real units: minutes)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f} minutes")
print(f"RMSE: {rmse:.2f} minutes")
print(f"R²: {r2:.4f}")

MAE: 78.62 minutes
RMSE: 99.55 minutes
R²: 0.9575


In [242]:
# Ensure same shape
y_test_arr = y_test.values.ravel()
y_pred_arr = np.array(y_pred)

sla_windows = [30, 60, 120]

print("Baseline SLA metrics:\n")
for window in sla_windows:
    baseline_sla = np.mean(np.abs(y_test_arr) <= window)
    print(f"SLA ±{window} min: {baseline_sla:.2%}")

print("\n\nML SLA metrics:\n")
for window in sla_windows:
    ml_sla = np.mean(np.abs(y_test_arr - y_pred_arr) <= window)
    print(f"SLA ±{window} min: {ml_sla:.2%}")

print("\n\nSLA Improvement:\n")
for window in sla_windows:
    baseline_sla = np.mean(np.abs(y_test_arr) <= window)
    ml_sla = np.mean(np.abs(y_test_arr - y_pred_arr) <= window)
    improvement = (ml_sla - baseline_sla) * 100
    print(f"SLA ±{window} min improved by {improvement:.2f} percentage")

Baseline SLA metrics:

SLA ±30 min: 6.80%
SLA ±60 min: 18.96%
SLA ±120 min: 28.04%


ML SLA metrics:

SLA ±30 min: 24.44%
SLA ±60 min: 46.40%
SLA ±120 min: 77.86%


SLA Improvement:

SLA ±30 min improved by 17.64 percentage
SLA ±60 min improved by 27.44 percentage
SLA ±120 min improved by 49.82 percentage


When moving from linear to tree-based models, we observed diminishing improvements, suggesting limited exploitable non-linear structure in the available features. Random Forest marginally improved SLA over Linear Regression, indicating limited non-linear structure in the data. Since further gains were unlikely without new features, I stopped tuning and focused on understanding error drivers instead.

In [247]:
# Feature importance from the trained RF

# Get feature importances
importances = rf_model.feature_importances_

# Feature names MUST match training data
feature_names = x_train_scaled_df.columns

# Build DataFrame
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
})

# Sort by importance
feature_importance_df = feature_importance_df.sort_values(
    by='importance',
    ascending=False
).reset_index(drop=True)

# Convert to percentage
feature_importance_df['importance_pct'] = (
    feature_importance_df['importance'] * 100
).round(2)

# Cumulative importance
feature_importance_df['cumulative_pct'] = (
    feature_importance_df['importance_pct'].cumsum()
)

feature_importance_df

,feature,importance,importance_pct,cumulative_pct
0,delivery_mode_standard,0.617284,61.73,61.73
1,delivery_mode_two day,0.249865,24.99,86.72
2,distance_km,0.061436,6.14,92.86
3,delivery_mode_same day,0.023030,2.30,95.16
4,weather_condition_stormy,0.021342,2.13,97.29
5,weather_condition_rainy,0.014255,1.43,98.72
6,weather_condition_foggy,0.005625,0.56,99.28
7,package_weight_kg,0.003655,0.37,99.65
8,region_north,0.000237,0.02,99.67
9,region_east,0.000211,0.02,99.69


“Random Forest feature importance analysis showed that ETA deviations are primarily driven by delivery mode, accounting for nearly 87% of the predictive signal, with distance and weather contributing marginally. The dominance of a small number of factors explains why increasing model complexity beyond linear models will yield diminishing improvements.”

In [250]:
# Build a test-level DataFrame for slicing

error_df = pd.DataFrame({
    'delivery_mode': x_test['delivery_mode'],  # original (not encoded)
    'y_true': y_test_arr,
    'y_pred': y_pred_arr
})

# Absolute error
error_df['abs_error'] = np.abs(error_df['y_true'] - error_df['y_pred'])

In [251]:
# SLA slicing by delivery_mode

sla_windows = [30, 60, 120]

sla_results = []

for mode in error_df['delivery_mode'].unique():
    subset = error_df[error_df['delivery_mode'] == mode]
    
    result = {'delivery_mode': mode}
    
    for window in sla_windows:
        result[f'SLA_±{window}_min'] = (
            np.mean(subset['abs_error'] <= window) * 100
        )
    
    sla_results.append(result)

sla_by_mode_df = pd.DataFrame(sla_results)
sla_by_mode_df

,delivery_mode,SLA_±30_min,SLA_±60_min,SLA_±120_min
0,standard,24.654190,46.948739,78.844589
1,express,22.997621,44.726408,75.971451
2,same day,25.315457,48.107256,77.760252
3,two day,24.798712,45.813205,78.904992


“Although delivery mode was the dominant feature, post-correction SLA metrics were consistent across all delivery modes. This showed that the model successfully removed delivery-mode bias, and that further improvements would require additional real-time features rather than more complex models.”

## Conclusion

This project demonstrated how ETA deviation can be modeled as a residual learning problem. Linear models captured most of the predictive signal, while tree-based models provided marginal improvements. Feature importance and error slicing revealed that delivery-mode bias was the dominant factor, and that remaining errors are driven by missing operational signals such as traffic and time-of-day. Further improvements would therefore require additional data rather than increased model complexity.